In [1]:
# !pip install pymupdf
# !pip install sentence-transformers
# !pip install faiss-cpu
# !pip install transformers
# !pip install torch
# !pip install fastapi uvicorn
# !pip install python-multipart
# !pip install rank-bm25
# !pip install numpy pandas

In [2]:
# !pip install pytesseract
# !pip install pdf2image
# !pip install pillow

In [3]:
from pdf2image import convert_from_path
import pytesseract

In [4]:
import fitz #pyMuPDF
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from rank_bm25 import BM25Okapi

C:\Users\Chidiebere\Documents\RAG1\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
pdf_path = "Nigeria-Data-Protection-Act-2023.pdf"

In [ ]:
# # Extract raw texts from PDF
# def extract_text_from_pdf(pdf_path):
#     document = fitz.open(pdf_path)

#     full_text = ""

#     for page_num in range(len(document)):
#         page = document[page_num]
#         text = page.get_text()

#         full_text += text

#     return full_text


# raw_text = extract_text_from_pdf(pdf_path)

# print(raw_text[:5000])
# # This codeblock didn't work because the source pdf file is not a digitally embedded text but image-based or scanned.

In [8]:
# Extract raw texts from PDF

def extract_text_with_ocr(pdf_path):
    pages = convert_from_path(pdf_path)

    full_text = ""

    for i, page in enumerate(pages):
        text = pytesseract.image_to_string(page)
        
        full_text += text

        print(f" processed page {i+1}")
        
    return full_text

raw_text = extract_text_with_ocr(pdf_path)
print(raw_text[:5000])

 processed page 1
 processed page 2
 processed page 3
 processed page 4
 processed page 5
 processed page 6
 processed page 7
 processed page 8
 processed page 9
 processed page 10
 processed page 11
 processed page 12
 processed page 13
 processed page 14
 processed page 15
 processed page 16
 processed page 17
 processed page 18
 processed page 19
 processed page 20
 processed page 21
 processed page 22
 processed page 23
 processed page 24
 processed page 25
 processed page 26
 processed page 27
 processed page 28
 processed page 29
 processed page 30
 processed page 31
 processed page 32
 processed page 33
 processed page 34
 processed page 35
 processed page 36
 processed page 37
 processed page 38
 processed page 39
 processed page 40
 processed page 41
 processed page 42
 processed page 43
 processed page 44
 processed page 45
 processed page 46
 processed page 47
NIGERIA DATA PROTECTION ACT, 2023

EXPLANATORY MEMORANDUM

This Act provides a legal framework for the protection of

In [9]:
# Text Cleaning

import re

def clean_text(text):

    # Remove excessive whitespace
    text = re.sub(r'\s+', " ", text)

    # Remove repeated newlines
    text = re.sub(r'\n+', "\n", text)

    # Strip leading and trailing spaces
    text = text.strip()

    return text

cleaned_text = clean_text(raw_text)

print(cleaned_text[:3000])

NIGERIA DATA PROTECTION ACT, 2023 EXPLANATORY MEMORANDUM This Act provides a legal framework for the protection of personal information and establishes the Nigeria Data Protection Commission for the regulation of the processing of personal information. Section: PART I— OBJECTIVES AND APPLICATION 1. Objectives NIGERIA DATA PROTECTION ACT, 2023 Arrangement of Sections 2. Application 3. PART Il — ESTABLISHMENT OF THE NIGERIA DATA PROTECTION COMMISSION AND ITS GOVERNING COUNCIL Establishment of the Nigeria Data Protection Commission SRN AUB 8. 24, 25. 26, 27. 4. 15. 16. 17. ai, 22, 23, Exemption of application > Functions of the Commission Powers of the Commission Independence of the Commission Establishment of the Governing Council of the Commission Appointment of members of the Council 10) 0. Tenure of members of the Council 11. Cessation of membership 12. Functions and powers of the Council 3. Conflict of interest PART Hi — APPOINTMENT OF THE NATIONAL COMMISSIONER, AND OTHER STAFF F THE

In [10]:
print(len(raw_text))

86049


## Chunking Strategy

##### Chunking determines Retrieval precision and recall, context quality and LLM hallucination rate

If **chunks are too small**, context becomes fragmented and important legal clauses become disconnected. And if **chunks are too large**, embeddings become diluted and retrieval becomes noisy.

We are using Simple chunking function

**Trade-off**:

Character-based chunking is simple.
Token-based chunking is more accurate.
Semantic chunking is even better but more complex.

In [11]:
# We use medium-large chunks because legal meaning depends heavily on surrounding context (700-1000 tokens)

def chunk_text(text, chunk_size=800, overlap=150):
    
    chunks = []
    
    start = 0
    
    while start < len(text):
        
        end = start + chunk_size
        
        chunk = text[start:end]
        
        chunks.append(chunk)
        
        
        # Overlap preserves context continuity.
        start += chunk_size - overlap
        
    return chunks

chunks = chunk_text(cleaned_text)
print(f"total chunks: {len(chunks)}")
print(chunks[0])

total chunks: 132
NIGERIA DATA PROTECTION ACT, 2023 EXPLANATORY MEMORANDUM This Act provides a legal framework for the protection of personal information and establishes the Nigeria Data Protection Commission for the regulation of the processing of personal information. Section: PART I— OBJECTIVES AND APPLICATION 1. Objectives NIGERIA DATA PROTECTION ACT, 2023 Arrangement of Sections 2. Application 3. PART Il — ESTABLISHMENT OF THE NIGERIA DATA PROTECTION COMMISSION AND ITS GOVERNING COUNCIL Establishment of the Nigeria Data Protection Commission SRN AUB 8. 24, 25. 26, 27. 4. 15. 16. 17. ai, 22, 23, Exemption of application > Functions of the Commission Powers of the Commission Independence of the Commission Establishment of the Governing Council of the Commission Appointment of members of the Council 10) 0


In [12]:
print(chunks[1])

 of the Commission Independence of the Commission Establishment of the Governing Council of the Commission Appointment of members of the Council 10) 0. Tenure of members of the Council 11. Cessation of membership 12. Functions and powers of the Council 3. Conflict of interest PART Hi — APPOINTMENT OF THE NATIONAL COMMISSIONER, AND OTHER STAFF F THE COMMISSION Appointment of the National Commissioner for the Commission Secretary ‘0 the Council Staff of the Commission Staff regu Pension, PART IV — F 19. 20. Funds of t ations and discipline INANCIAL PROVISIONS he Commission Expenditure of the Fund Power to borrow end accept gifts Account and audit Annual reports and estimates PART V — PRINCIPLES AND LAWFUL BASIS GOVERNING PROCESSING OF Principles PERSONAL DATA of personal data processing Lawf


### Embedding Generation

**Embeddings convert text into vectors**. Vectors capture semantic meaning which enables similarity search, semantic retrieval and context-aware querying.

In [13]:
# load embedding model

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7257.07it/s]


In [14]:
# Generate embeddings for all chunks, shape example:
# (number_of_chunks, embedding_dimension)

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(chunk_embeddings.shape)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.39it/s]

(132, 384)


### Building the Vector Database

FAISS - Facebook AI Similarity Search; performs efficient nearest-neighbor vector search.


Without FAISS, retrieval becomes extremely slow and linear search scales poorly.

In [15]:
# Get embedding dimension

embedding_dimension = chunk_embeddings.shape[1]

print(embedding_dimension)

384


In [17]:
# Create FAISS Index - IndexFlatL2

index = faiss.IndexFlatL2(embedding_dimension)

# Add embeddings into FAISS
index.add(chunk_embeddings)

print(f"total vectors in index: {index.ntotal}")

total vectors in index: 132


### Retriever Construction

The heart of RAG

In [18]:
# Build retrieval function

def retrieve(query, top_k=5):
    
    # Convert query into embedding
    query_embedding = embedding_model.encode([query])
    
    # Search nearest neighbours
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    
    for idx in indices[0]:
        results.append(chunks[idx])
        
    return results        

In [19]:
query = "What are the rights of data subjects under NDPA?"

results = retrieve(query)

for i, result in enumerate(results):
    print(f"\n === RESULT {i+1} ===\n")
    print(result)


 === RESULT 1 ===

al right to privacy ofa data subject. (2) Subject to the rights and freedoms under the Constitution and the limitations, the obligations under Part V, other than sections 24, 25, 32, and 40 of this Act, shall not apply to a data controller or data processor if the processing of personal data is — (a) carried out by a competent authority for the purposes of the prevention, investigation, detection, prosecution, or adjudication of a criminal offence or the execution of a criminal penalty, in accordance with any applicable law; (b) carried out by a competent authority for the purposes of prevention or control of a national public health emergency; (c) carried out by a competent authority, as is necessary for national security; (d) in respect of publication in the public interest, for journali

 === RESULT 2 ===

nduct issued by the Commission made under the provisions of this Act. PART VI-— RIGHTS OF A DATA SUBJECT 34. (1) A data subject has the right to obtain from a 

### Hybrid Search

Hybrid retrieval combines semantic search and keyword search

In [20]:
# Prepare BM25 corpus

tokenized_chunks = [chunk.split(" ") for chunk in chunks]

bm25 = BM25Okapi(tokenized_chunks)

In [27]:
# Hybrid retrieval function

def hybrid_retrieve(query, top_k=5):
    
    # Dense retrieval
    query_embedding = embedding_model.encode([query])
    
    # FAISS expects float32
    query_embedding = query_embedding.astype("float32")
    
    distances, dense_indices = index.search(query_embedding, top_k)
    
    dense_results = [chunks[idx] for idx in dense_indices[0]]
    
    # BM25 retrieval
    tokenized_query = query.split(" ")
    
    bm25_scores = bm25.get_scores(tokenized_query)
    
    bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]
    
    bm25_results = [chunks[idx] for idx in bm25_indices]
    
    # Merged Result
    merged = list(dict.fromkeys(dense_results + bm25_results))
    
    return merged[:top_k]

In [28]:
query = "Penalties for unlawful processing of personal data"

results = hybrid_retrieve(query)

for i, result in enumerate(results):
    print(f"\n=== HYBRID RESULT {i+1} ===\n")
    print(result)


=== HYBRID RESULT 1 ===

ther processing, (iv) how the personal data has been collected, and (v) the existence of appropriate safeguards; and (b) further processing for archiving purposes in the public interest, scientific, historical research purposes, or statistical purposes shall not be considered to be incompatible with the initial purposes. Principles of Personal data Processing 25. (1) Without prejudice to the principles set out in this Act, data processing shall be lawful, where — (a) the data subject has given and not withdrawn consent for the specific purpose or purposes for which personal data is to be processed; or (b) the processing is necessary — (i) for the performance of a contract to which the data subject is a party or to take steps at the request of the data subject prior to entering into a contr

=== HYBRID RESULT 2 ===

it Annual reports and estimates PART V — PRINCIPLES AND LAWFUL BASIS GOVERNING PROCESSING OF Principles PERSONAL DATA of personal data processing L

### Large Language Model Integration

HuggingFace Pipeline is open source

In [30]:
# # Using HuggingFace text-generation pipeline

# text_generator = pipeline(
#     "text-generation",
#     model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# )

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 6206.56it/s]


In [ ]:
# # Using generationConfig
# from transformers import pipeline, GenerationConfig

# generation_config = GenerationConfig(
#     max_new_tokens=200,
#     do_sample=False
# )

# text_generator = pipeline(
#     "text-generation",
#     model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# )

In [ ]:
# # Then generate like this

# response = text_generator(
#     prompt,
#     generation_config=generation_config
# )

### Prompt engineering function

Prompt engineering strongly affects hallucination, grounding, answer quality and citation behavior

In [39]:
def build_prompt(query, contexts):
    context_text = "\n\n".join(contexts)
    
    prompt = f"""
You are a legal assistant specialized in the Nigerian Data Protection Act 2023.

Answer ONLY using the provided context.

If the answer is not in the context, say:
'I could not find the answer in the provided document.'

Context:
{context_text}

Question:
{query}

Answer:
"""
    return prompt

In [40]:
# Generate response

query = "What are the obligations of data controllers?"

contexts = hybrid_retrieve(query)

prompt = build_prompt(query, contexts)

In [41]:
from transformers import pipeline, GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=200,
    do_sample=False
)

text_generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

response = text_generator(
    prompt,
    generation_config=generation_config,
    clean_up_tokenization_spaces=False
)

print(response[0]["generated_text"])

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 5334.31it/s]
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a legal assistant specialized in the Nigerian Data Protection Act 2023.

Answer ONLY using the provided context.

If the answer is not in the context, say:
'I could not find the answer in the provided document.'

Context:
nd (d) the measures envisaged to address the risks, safeguards, security measures and mechanisms to ensure the protection of personal data, taking into account the tights and legitimate interests of a data subject and other persons concerned. 29. (1) Where a data controller engages the services of a data processor, or a data processor engages the services of another data processor, the data controller or data processor engaging another shall ensure that the engaged data processor — (a) complies with the principles and obligations set out in this Act as applicable to the data controller; (b) assists the data controller or data processor, as the case may be, by the use of appropriate technical and organisational measures, in the fulfilment of the data controlle

### Hallucination Reduction

In [42]:
# Sometimes retrieved chunks are irrelevant so we add retrieval confidence threshold.

def retrieve_with_scores(query, top_k=5):
    query_embedding = embedding_model.encode([query])
    
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    
    
    for score, idx in zip(distances[0], indices[0]):
        results.append({
            "chunk": chunks[idx],
            "distance": float(score)
            
        })
        
    return results

### Meta data enrichment

Production RAG systems store page numbers, section titles, dates, document IDs, and source filenames.

This improves filtering, explainability, citations and compliance.

In [47]:
# chunk objects with metadata

chunk_objects = []

for idx, chunk in enumerate(chunks):
    chunk_objects.append({
        "chunk_id": idx,
        "text": chunk,
        "source": "NDPA_2023",
    })

chunk_objects[:2]

[{'chunk_id': 0, 'text': '0', 'source': 'NDPA_2023'}]

### Saving the Vector Database

In [48]:
# Save FAISS index

faiss.write_index(index, "ndpa_faiss.index")

In [49]:
# Reload FAISS index

loaded_index = faiss.read_index("ndpa_faiss.index")

print(loaded_index.ntotal)

132


### FastAPI Deployment